моя нейронная сеть с одним скрытым слоем для распознавая цифр со свтроенного датасета sklearn

In [54]:
import pandas as pd
import numpy as np
from sklearn.datasets import load_digits
from sklearn.model_selection import train_test_split
import matplotlib.pyplot as plt

In [55]:
digits = load_digits()
y = digits.target
X = digits.data#.reshape(-1, 64, 1)
print(X.shape)
print(y.shape)
print(X[0].shape)
print(len(X))

y = np.eye(10)[y]


(1797, 64)
(1797,)
(64,)
1797


In [56]:
def draw(X, x, y):
    matrix = X.reshape(x,y)
    plt.imshow(matrix, cmap='gray')
    plt.axis('off')
    plt.show()


# Алгоритм обратного распространения ошибки (Backpropagation) и методы оптимизации

Данный документ содержит математическое описание алгоритма Backpropagation на основе конспекта.

---

## 1. Математическое описание алгоритма

### Переменные и обозначения:
* $l$ — индекс текущего слоя ($l = 1, \dots, L$).
* $h$ — индекс нейрона в слое.
* $w_{kh}^l$ — вес связи между $k$-м нейроном слоя $(l-1)$ и $h$-м нейроном слоя $l$.
* $S_{ih}^l$ — взвешенная сумма (вход активации) для $h$-го нейрона слоя $l$ на объекте $x_i$.
* $x_{ih}^l$ — выходное значение (активация) $h$-го нейрона слоя $l$.
* $z_{ih}^l$ — производная функции активации текущего слоя.
* $\varepsilon_{hi}^l$ — локальный градиент ошибки на слое $l$.
* $\eta$ — скорость обучения (learning rate).

### Прямой ход (Forward Pass):
Выполняется последовательно для всех слоев $l = 1, \dots, L$:

$$S_{ih}^l = \sum_{k=0}^{H_{l-1}} w_{kh}^l x_{ik}^{l-1}$$

$$x_{ih}^l = \sigma(S_{ih}^l)$$

$$z_{ih}^l = \sigma'(S_{ih}^l) = \sigma(S_{ih}^l) \cdot (1 - \sigma(S_{ih}^l))$$

Градиент функции потерь по выходу сети:

$$\varepsilon_{hi}^L = \frac{\partial L_i(w)}{\partial x_{ih}^L}$$

### Обратный ход (Backward Pass):
Выполняется последовательно для слоев в обратном порядке от $l = L, \dots, 2$:

$$\varepsilon_{ik}^{l-1} = \sum_{h=0}^{H_l} \varepsilon_{ih}^l z_{ih}^l w_{kh}^l$$

### Градиентный шаг (Weight Update):
$$\Delta w_{kh}^l = -\eta \cdot \varepsilon_{ih}^l z_{ih}^l x_{ik}^{l-1}$$

$$w_{kh}^l := w_{kh}^l + \Delta w_{kh}^l$$

---

## 2. Эвристики и продвинутые методы оптимизации

Когда стандартный градиентный спуск нестабилен, применяются адаптивные методы:

### RMSProp (Running Mean Square)
Выравнивает скорости изменения весов с помощью скользящего среднего по итерациям. Ускоряет обучение по весам, которые до этого мало изменялись ($\alpha \approx \frac{1}{1-\gamma}$):

$$G := \alpha G + (1 - \alpha) L'_i(w) \odot L'_i(w)$$

$$w := w - \eta \cdot L'_i(w) \oslash (\sqrt{G} + \varepsilon)$$

### Gradient Clipping (Ограничение градиента)
Защищает от проблемы взрыва градиентов. Если норма вектора градиента превышает порог $\theta$:

$$\text{Если } \|g\| > \theta \implies g := \frac{g \cdot \theta}{\|g\|}$$


In [81]:
class MyNeurNet:
    def __init__(self, arch, learning_rate=0.01, n_iters=1000, batch_size= 32):
        np.random.seed(42)
        self.learning_rate=learning_rate
        self.n_iters=n_iters
        self.W = []
        self.B = []
        self.batch_size = batch_size
        for i in range(len(arch)-1):
            curr_lay = arch[i]
            next_lay = arch[i+1]
            
            scale = np.sqrt(2.0 / curr_lay) * 0.1
            curr_w = np.random.randn(curr_lay, next_lay) * scale
            curr_b = np.zeros((1, next_lay))

            self.W.append(curr_w)
            self.B.append(curr_b)
    def ReLU(self, x):
        return np.maximum(0, x)
    
    def der_ReLU(self, x):
        return (x > 0).astype(float) 


    def sigmoid(self, z):
        return 1/(1+np.exp(-z))
    
    def der_sigmoid(self, z):
        return self.sigmoid(z)*(1-self.sigmoid(z))
    
    def predict_for_lay(self, x, w, b):
        return self.ReLU(x @ w + b)
    
    def predict_for_last_lay(self, x, w, b):
        return self.sigmoid(x @ w + b)
    
    def predict_for_layers(self, X):
        S = []
        A = []
        Z = []
        A.append(X)
        for i in range(len(self.W) - 1):
            X = X @ self.W[i] + self.B[i]
            S.append(X)
            Z.append(self.der_ReLU(X))
            X = self.ReLU(X)
            A.append(X)
        X = X @ self.W[len(self.W) - 1] + self.B[len(self.W) - 1]
        S.append(X)
        Z.append(self.der_sigmoid(X))
        X = self.sigmoid(X)
        A.append(X)
        return S, A, Z
    
    def predict(self, x):
        for i in range(len(self.W) - 1):
            x = self.predict_for_lay(x, self.W[i], self.B[i])
        x = self.predict_for_last_lay(x, self.W[len(self.W) - 1], self.B[len(self.W) - 1])
        return x

    def lay_loss_on_object(self, x, y):
        x = self.predict(x)
        return x - y

    def sum_loss_on_object(self, x, y):
        x_pred = self.predict(x)
        loss = 0.0
        for i in range(len(y)):
            loss += ((x_pred[i] - y[i])**2)*0.5
        return loss

    def fit(self, X, y):
        for i in range(self.n_iters):
            self.batch_size
            random_ind = np.random.choice(X.shape[0], self.batch_size, replace=False)
            X_batch = X[random_ind]
            y_batch = y[random_ind]
            S = []  #результат w[i]x[i] + b
            A = []  #sigmoid(S)
            Z = []  #dersigmoid(S)
            S, A, Z = self.predict_for_layers(X_batch)
            loss = A[len(self.W)] - y_batch
            for j in range(len(self.W) - 1, -1, -1):
                # print(f"X_batch: {X_batch.shape}")
                # print(f"y_batch: {y_batch.shape}")
                # print(f"loss: {loss.shape}")
                # print(f"Z[j]: {Z[j].shape}")
                # print(f"A[j]: {A[j].shape}")
                # print(f"B[j]: {self.B[j].shape}")
                # print(f"W[j]: {self.W[j].shape}")
                delta = loss * Z[j]
                dw = (A[j].T @ delta) / self.batch_size
                db = np.sum(delta, axis=0, keepdims=True) / self.batch_size
                self.W[j] -= self.learning_rate * dw
                self.B[j] -= self.learning_rate * db
                if j > 0:
                    loss = delta @ self.W[j].T


In [91]:
mymodel = MyNeurNet (
    arch=[64, 18, 10],
    learning_rate= 0.05,
    n_iters=800
)
X_train, X_test, y_train, y_test = train_test_split(X, y, random_state=42)
mymodel.fit(X_train, y_train)
correct = 0
for i in range(len(y_test)):
    pred_pob = mymodel.predict(X_test[i])
    pred = np.argmax(pred_pob)
    if y_test[i][pred] == 1:
        correct += 1
    # else:
    #     print(f"Пред: {pred}| Факт: {np.argmax(y_test[i])}")
    #     draw(X_test[i])

print(f"Всего: {len(y_test)}")
print(f"Правильных: {correct}")

Всего: 450
Правильных: 430


теперь картинки 28 на 28

In [59]:
df_train = pd.read_csv('pic_train.csv')
df_test = pd.read_csv('pic_test.csv')
df_train.head(1)
y_train_pic = df_train['label'].values
X_train_pic = df_train.drop(columns=['label']).values / 255.0
y_train_pic = np.eye(10)[y_train_pic]


y_test_pic = df_test['label'].values
X_test_pic = df_test.drop(columns=['label']).values / 255.0
y_test_pic = np.eye(10)[y_test_pic]

print(f"Train shapes: X={X_train_pic.shape}, y={y_train_pic.shape}")
print(f"Test shapes:  X={X_test_pic.shape}, y={y_test_pic.shape}")


Train shapes: X=(60000, 784), y=(60000, 10)
Test shapes:  X=(10000, 784), y=(10000, 10)


In [101]:
mymodel2 = MyNeurNet (
    [784, 200, 80, 10],
    learning_rate=0.15,
    n_iters=5000,
    batch_size=72
)
mymodel2.fit(X_train_pic, y_train_pic)
correct2 = 0
for i in range(len(y_test_pic)):
    pred_pob = mymodel2.predict(X_test_pic[i])
    pred = np.argmax(pred_pob)
    if y_test_pic[i][pred] == 1:
        correct2 += 1
    # else:
    #     print(f"Пред: {pred}| Факт: {np.argmax(y_test_pic[i])}")
    #     draw(X_test_pic[i], 28,28)
print(f"Всего: {len(y_test_pic)}")
print(f"Правильных: {correct2}")

Всего: 10000
Правильных: 9157
